# Face Recognition System - Complete Course

## Project Overview

This notebook provides a comprehensive walkthrough of building a face recognition system using modern ML techniques. You'll learn:

1. **Face Detection**: Locating faces in images and video streams using HOG (Histogram of Oriented Gradients) and CNN models
2. **Face Encoding**: Converting face images into high-dimensional embeddings (128-D vectors) for comparison
3. **Face Recognition**: Matching captured faces against a database of known faces
4. **Persistence**: Efficiently storing and loading face encodings

### Key Libraries
- `face_recognition`: High-level wrapper around dlib for face detection, encoding, and comparison
- `opencv-python (cv2)`: Computer vision library for image manipulation and video capture
- `numpy`: Numerical computing for array operations
- `matplotlib`: Visualization of images and results

This curriculum is designed for practitioners with ML experience who want a solid understanding of the practical implementation.

## Phase 1: Setup & Dependencies

In [ ]:
# Import primary libraries for face recognition workflow
import face_recognition as fr  # High-level face detection and encoding API
import cv2                      # OpenCV: image processing and camera capture
import numpy as np              # Numerical operations and array handling
import matplotlib.pyplot as plt  # Visualization
from time import sleep          # For frame delays during capture
import os                       # File system operations
import pickle                   # Serialization of face encodings

## Phase 2: Understanding Face Detection

### Concept: Face Detection Models

Face detection is the process of locating faces in an image. Two primary models are used:

- **HOG (Histogram of Oriented Gradients)**: Fast, CPU-friendly model that compares edge orientations to learned patterns. Effective for frontal faces.
- **CNN (Convolutional Neural Networks)**: Slower but more accurate deep learning model. Better for varied face angles and lighting conditions.

The `face_recognition` library uses dlib's implementations. Face locations are returned as tuples `(top, right, bottom, left)` in pixel coordinates.

In [ ]:
def show_image_with_faces(img, face_locations):
    """
    Visualize detected faces by drawing rectangles around them.
    
    Args:
        img: Input image (numpy array, BGR format from OpenCV)
        face_locations: List of tuples (top, right, bottom, left) in pixel coordinates
    """
    for index, face_location in enumerate(face_locations):
        print(f"Found face {index} at location: {face_location}")
        # Unpack face location coordinates
        t, r, b, l = face_location
        # Draw bounding box: (image, top-left, bottom-right, color in BGR, thickness)
        img = cv2.rectangle(img, (l, t), (r, b), (250, 5, 5), 1)
    
    # Display result
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
# Initialize data structure to store captured faces
faces_in_frame = []  # Will hold cropped face images from camera stream

## Phase 3: Capturing & Processing Video

### Concept: Video Acquisition Pipeline

Capturing video from a camera involves:
1. **Initialization**: Open camera stream at specified index (typically 0 for default camera)
2. **Configuration**: Set resolution to balance speed and quality
3. **Frame Loop**: Continuously read frames, detect faces, and process results
4. **Preprocessing**: Normalize frame orientation and symmetry for consistent detection
5. **Visualization**: Draw detection results and display live feed
6. **Cleanup**: Release resources when done

### Frame Processing Steps
- **Rotation**: Handle camera orientation (landscape/portrait)
- **Flip**: Mirror horizontal axis for natural selfie perspective
- **Detection**: Run HOG model on each frame (computationally efficient)
- **Cropping**: Extract individual face regions for encoding

In [ ]:
# Note: This cell requires camera access and is interactive. 
# Uncomment to run with your camera device.

"""
# Set camera device index (usually 0 for default camera)
camera_index = 0

# Initialize video capture object
cap = cv2.VideoCapture(camera_index)

# Configure camera resolution for balance between speed and quality
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Verify camera successfully opened
if not cap.isOpened():
    print(f"Error: Could not open video device at index {camera_index}")
    exit()

# Main capture loop
while True:
    # Read frame from camera
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture image")
        break

    # Preprocess frame: handle device orientation
    # Many phone cameras return rotated frames due to device orientation
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)      # First rotation
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)      # Second rotation (180 total)
    frame = cv2.flip(frame, 1)  # Horizontal flip for selfie perspective

    # Detect faces using HOG model (fast, CPU-friendly)
    face_locations = fr.face_locations(frame, model="hog")
    
    if face_locations:
        print(f"Found {len(face_locations)} face(s) in the current frame.")
        # Pause briefly to allow viewing the frame
        sleep(5)
        
        # Create copy of frame for face extraction
        user_img = frame.copy()
        
        # Crop and store individual face regions
        for (top, right, bottom, left) in face_locations:
            # Extract region: [rows (top:bottom), cols (left:right)]
            cropped_face = user_img[top:bottom, left:right]
            faces_in_frame.append(cropped_face)
        
        # Draw detection bounding boxes on frame for visualization
        for (top, right, bottom, left) in face_locations:
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)

    # Display live feed with detections
    cv2.imshow("Phone Camera Stream", frame)

    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup: release camera and close windows
cap.release()
cv2.destroyAllWindows()
"""

print("Camera capture code is provided above (commented out for interactive use).")
print("Uncomment and run to capture faces from your camera.")

In [ ]:
# Display captured faces from the stream
# This loop visualizes each face that was extracted during camera capture

"""
for index, face in enumerate(faces_in_frame):
    # Convert BGR (OpenCV default) to RGB for matplotlib
    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    plt.imshow(face_rgb)
    plt.axis("off")
    plt.title(f"Face {index}")
    plt.show()
"""

print("Uncomment above to display captured faces.")

## Phase 4: Face Encoding & Recognition

### Concept: Face Embeddings

Face encoding converts a face image into a **128-dimensional embedding vector**. This vector:
- Captures the essential facial features in a normalized space
- Enables efficient similarity comparison between faces
- Is generated using dlib's deep neural network trained on millions of faces

**Key insight**: The distance between two face encodings indicates similarity:
- Distance < 0.6: Likely the same person (tunable threshold)
- Distance > 0.6: Different people

### Workflow
1. Load reference images of known individuals
2. Generate encodings for each reference image
3. Store encoded database for fast comparison
4. Compare new faces against database using distance metrics

In [ ]:
# Load reference images from the 'images' directory
# Expected structure: images/
#                      ├── person1.jpg
#                      ├── person2.jpg
#                      └── ...

# Check if images directory exists
if os.path.exists("images"):
    # Get list of image files
    image_files = os.listdir("images")
    print(f"Found {len(image_files)} reference image(s)")
    
    # Extract person IDs from filenames (remove extension)
    ids = [os.path.splitext(image)[0] for image in image_files]
    
    # Load images using OpenCV (returns BGR format)
    images = [cv2.imread(f"images/{image}") for image in image_files]
    
    # Display first reference image as verification
    if len(images) > 0:
        print(f"First reference image shape: {images[0].shape}")
else:
    print("Please create an 'images' directory with reference face images.")

In [ ]:
def face_encoding(face):
    """
    Generate a 128-dimensional face encoding vector.
    
    Args:
        face: Face image (numpy array, BGR format from OpenCV)
        
    Returns:
        face_encoding: 128-D numpy array representing the face
        
    Note: This function assumes a single face in the input image.
          If multiple faces are present, only the first is encoded.
    """
    # Convert from BGR (OpenCV) to RGB (face_recognition expects RGB)
    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    
    # Generate encoding using dlib's CNN model
    # Returns list of encodings (one per detected face)
    encodings = fr.face_encodings(face_rgb)
    
    # Return first encoding (assumes single face per image)
    if len(encodings) > 0:
        return encodings[0]
    else:
        print("Warning: No face detected in image")
        return None

In [ ]:
# Generate encodings for all reference images
# This creates our face database that we'll later use for recognition

if os.path.exists("images") and 'images' in locals():
    print("Generating face encodings for reference images...")
    face_encodings = []
    
    for i, face in enumerate(images):
        encoding = face_encoding(face)
        if encoding is not None:
            face_encodings.append(encoding)
            print(f"Encoded face {i}/{len(images)} ({ids[i]}) - Shape: {encoding.shape}")
        else:
            print(f"Failed to encode face {i} ({ids[i]})")
    
    print(f"\nTotal encodings generated: {len(face_encodings)}")
else:
    print("Reference images not loaded. Please run previous cells.")

In [ ]:
# Inspect a sample encoding to understand the data structure
if 'face_encodings' in locals() and len(face_encodings) > 0:
    print("Sample face encoding (first reference image):")
    print(f"Type: {type(face_encodings[0])}")
    print(f"Shape: {face_encodings[0].shape}")
    print(f"First 10 values: {face_encodings[0][:10]}")
    print(f"\nThis 128-dimensional vector uniquely represents the person's facial features.")
else:
    print("No face encodings available.")

## Phase 5: Persistence & Serialization

### Concept: Storing Face Encodings

Rather than re-computing face encodings every time the system runs, we serialize them to disk:

- **Format**: Python pickle (binary serialization)
- **Structure**: List of tuples `[(person_id, encoding_vector), ...]`
- **Efficiency**: Avoids expensive re-encoding operations; fast at runtime
- **Trade-off**: Pickle is Python-specific; use JSON/protobuf for cross-language compatibility

### File: `faces.p`
Contains a serialized dictionary mapping person IDs to their 128-D face encodings.

In [ ]:
# Save face encodings to disk for later use
# This avoids recomputing encodings in every run

if 'face_encodings' in locals() and 'ids' in locals() and len(face_encodings) > 0:
    # Create list of (person_id, encoding) tuples
    faces_info = list(zip(ids, face_encodings))
    
    # Serialize to binary pickle format
    with open("faces.p", "wb") as f:
        pickle.dump(faces_info, f)
    
    print(f"Saved {len(faces_info)} face encodings to 'faces.p'")
    print(f"File size: {os.path.getsize('faces.p')} bytes")
else:
    print("Face encodings not available. Run Phase 4 cells first.")

In [ ]:
# Load face encodings from disk
# This demonstrates efficient retrieval of the pre-computed database

if os.path.exists("faces.p"):
    with open("faces.p", "rb") as f:
        loaded_faces_info = pickle.load(f)
    
    print(f"Loaded {len(loaded_faces_info)} face encodings from 'faces.p'")
    print(f"\nDatabase contents:")
    for person_id, encoding in loaded_faces_info:
        print(f"  - {person_id}: {encoding.shape} vector")
else:
    print("File 'faces.p' not found. Save encodings first (see previous cell).")

## Summary

### What You've Learned

1. **Face Detection**: Using HOG and CNN models to locate faces in images
2. **Video Processing**: Capturing and preprocessing camera frames
3. **Face Encoding**: Converting 2D face images to 128-D embedding vectors
4. **Face Database**: Building and persisting a reference database of known faces

### Next Steps

To complete a full recognition system, you would:
1. **Compare Encodings**: Use distance metrics (e.g., Euclidean) to find the closest match
2. **Apply Thresholds**: Set tolerance for same-person matching (typically 0.4-0.6)
3. **Handle Edge Cases**: Deal with unknown faces, poor quality images, or multiple matches
4. **Optimize Performance**: Use GPU acceleration for encoding large batches
5. **Deploy**: Package as an API or edge application

### Key Libraries & References
- `face_recognition` documentation: https://github.com/ageitgey/face_recognition
- dlib C++ library: http://dlib.net/
- OpenCV documentation: https://docs.opencv.org/

---

**Course completed!** You now have a solid foundation in practical face recognition systems.